# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) dataset using the `mlcroissant` library.### Dataset SourceCroissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure mlcroissant is installed!pip install mlcroissant

## 1. Data LoadingLoad dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pd# Croissant schema URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'# Load the dataset using mlcroissantdataset = mlc.Dataset(croissant_url)metadata = dataset.metadata# Print metadata summaryprint(f"{metadata.name}\n{metadata.description}\n")print(f"Published: {metadata.datePublished}")print(f"Version: {metadata.version}")print(f"Identifier: {metadata.identifier}")print(f"License: {metadata.license}")

## 2. Data OverviewExamine available record sets, their `@id`s, and the `@id`s of their fields.

In [ ]:
# List all record sets (tables/relations) defined in this datasetrecord_sets = list(dataset.record_sets)print("Available record sets and their field IDs:")for rset in record_sets:    print(f"- RecordSet @id: {rset['@id']}")    fields = rset.get('field', [])    # If one field, ensure it's a list    if isinstance(fields, dict):        fields = [fields]    field_ids = [field['@id'] if isinstance(field, dict) else str(field) for field in fields]    print("  Fields:")    for fid in field_ids:        print(f"    - Field @id: {fid}")    print()

## 3. Data ExtractionLoad data from a record set into a DataFrame. All record set and field selection is by their `@id` fields.*Note: If there are multiple record sets, you may repeat this for each one as needed.*

In [ ]:
# Extract all records from each record set into DataFrames, keyed by record set @iddataframes = {}for rset in record_sets:    recset_id = rset['@id']    print(f"Loading records for RecordSet {recset_id} ...")    # Each record is a dict of field-value for the given row    records = list(dataset.records(record_set=recset_id))    df = pd.DataFrame(records)    dataframes[recset_id] = df    print(f"  Columns: {df.columns.tolist()}")    # Show sample    display(df.head())

## 4. Exploratory Data Analysis (EDA)Apply basic data processing: filtering, normalization, and grouping, referencing field names by their `@id`.

In [ ]:
# For demonstration, select the first record set and try numeric field operationsif record_sets:    record_set_id = record_sets[0]['@id']    df = dataframes[record_set_id]    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()    # Attempt to select the first numeric field for demo purposes    if numeric_candidates:        numeric_field = numeric_candidates[0]   # This is the @id of the field        print(f"Selected numeric field: {numeric_field}")        threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field].mean()) else 0        filtered_df = df[df[numeric_field] > threshold]        print(f"Filtered rows with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} rows\n")        display(filtered_df.head())        # Normalization        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()        print(f"First 5 rows with normalized {numeric_field}:")        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())        # Try grouping by a categorical field        group_candidates = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]        if group_candidates:            group_field = group_candidates[0]            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")            print(f"Grouped mean {numeric_field} by {group_field}:")            display(grouped_df.head())        else:            print("No suitable categorical field for grouping found.")    else:        print("No numeric fields found in this record set.")else:    print("No record sets found in the Croissant metadata.")

## 5. VisualizationVisualize distribution of a numeric field, or other relationships between fields.*Note: Here we create a histogram and boxplot of the chosen numeric field if available.*

In [ ]:
import matplotlib.pyplot as pltif record_sets and 'numeric_field' in locals():    plt.figure(figsize=(12,4))    plt.subplot(1,2,1)    df[numeric_field].hist(bins=30)    plt.title(f"Histogram of {numeric_field}")    plt.xlabel(numeric_field)    plt.ylabel('Count')    plt.subplot(1,2,2)    df.boxplot(column=numeric_field)    plt.title(f"Boxplot of {numeric_field}")    plt.tight_layout()    plt.show()

## 6. ConclusionIn this notebook, we've loaded, explored, and visualized the FAIR^2 dataset using the `mlcroissant` library.Key steps demonstrated:- Loading dataset and metadata from a Croissant schema URL- Listing and loading record sets, referencing all structures and data fields by their `@id`- Performing basic filtering, normalization, and grouping using pandas- Creating visualizations for quick data insightsYou can adapt this workflow for any dataset specified by a Croissant schema. For further analysis, consult the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/) and extend this notebook with your scientific questions!